# Chapter 05 — Production Performance, Streaming Engine & Observability

Welcome to **Chapter 05** of the **Google Cloud Dataflow Workshop**!

In this chapter, you will master production performance tuning and operational observability:
1. **Dataflow Streaming Engine & Dataflow Shuffle**: Offloading shuffle and state storage to Google-managed services.
2. **Diagnosing System Lag vs. Data Freshness**: Identifying throughput deficits versus upstream ingestion pauses.
3. **Horizontal Autoscaling Optimization**: Balancing cost and responsiveness with `minNumWorkers` and `maxNumWorkers`.
4. **Fine-Grained Step Profiling**: Embedding `Metrics.distribution` and `Metrics.counter` in Beam `DoFns`.
5. **Automated SLA Alerting**: Deploying Cloud Monitoring Alert Policies and operational dashboards.

## 1. Environment Verification & Kernel Selection

Ensure that your notebook kernel is attached to the project's **`.venv`** managed by **`uv`**:
- Check the top-right kernel selector and select **`Python (dataflow-workshops)`**.

Run the cell below to verify your environment:

In [1]:
import os
import sys

print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is running inside the project's .venv!


✅ Apache Beam Version: 2.77.0


## 2. Architecture & Visual DAG

Here is the architecture for this chapter, illustrating the pipeline topology, Streaming Engine offloading, and Cloud Monitoring alert pipeline:

![Chapter 05 Streaming Engine & Monitoring DAG](../artifacts/05_streaming_engine_and_monitoring_dag.png)

<details>
<summary><b>Click to expand Mermaid source code</b></summary>

```mermaid
flowchart TD
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef engine fill:#f3e8fd,stroke:#9334e6,stroke-width:2px,color:#6b21a8,font-weight:bold;
    classDef alert fill:#fee2e2,stroke:#ef4444,stroke-width:2px,color:#991b1b,font-weight:bold;

    INPUT[benchmark_telemetry.json]:::pcoll --> PARSE[ParseAndBenchmark]:::ptrans
    PARSE --> GBK[GroupByCategory]:::ptrans
    GBK -. Offload Shuffle .-> STREAMING_ENGINE[Streaming Engine]:::engine
    GBK --> AGG[AggregateKPIs]:::ptrans
    AGG --> BQ_SINK[BigQuery benchmark_category_metrics]:::sink
    PARSE -. Metrics .-> METRIC_LAG[System Lag Metric]:::alert
    METRIC_LAG --> ALERT[Cloud Monitoring Alert]:::alert
```
</details>

## 3. Inspecting the Benchmark Telemetry Dataset

Let's inspect the high-volume benchmark dataset (`data/benchmark_telemetry.json`):

In [2]:
from pathlib import Path
import json

records = []
with open("data/benchmark_telemetry.json") as f:
    for i, line in enumerate(f):
        if i < 5:
            records.append(json.loads(line))

print("Sample Benchmark Records:")
for r in records:
    print(json.dumps(r))

Sample Benchmark Records:
{"record_id": "rec-000000", "user_id": "user_024", "category": "software", "amount": 116.91, "payload_bytes": 1911, "timestamp": "2026-10-27T14:00:03+00:00"}
{"record_id": "rec-000001", "user_id": "user_023", "category": "groceries", "amount": 311.04, "payload_bytes": 1784, "timestamp": "2026-10-27T14:00:05+00:00"}
{"record_id": "rec-000002", "user_id": "user_002", "category": "groceries", "amount": 23.39, "payload_bytes": 227, "timestamp": "2026-10-27T14:00:04+00:00"}
{"record_id": "rec-000003", "user_id": "user_024", "category": "groceries", "amount": 425.33, "payload_bytes": 181, "timestamp": "2026-10-27T14:00:10+00:00"}
{"record_id": "rec-000004", "user_id": "user_040", "category": "apparel", "amount": 192.1, "payload_bytes": 1096, "timestamp": "2026-10-27T14:00:09+00:00"}


## 4. Concept 1: Step Latency Instrumentation with Beam Metrics

Apache Beam allows embedding `Metrics.distribution` and `Metrics.counter` directly into `DoFn` steps.
This provides exact min, max, count, and mean execution latencies per transform step without external tracing tools:

Let's test this in a minimal Beam pipeline:

In [3]:
import time
import apache_beam as beam
from apache_beam.metrics import Metrics

class InstrumentedDoFn(beam.DoFn):
    def __init__(self):
        self.count_metric = Metrics.counter(self.__class__, "processed_elements")
        self.latency_metric = Metrics.distribution(self.__class__, "latency_ms")

    def process(self, element):
        t0 = time.perf_counter()
        # Simulate light processing
        result = element * 2
        elapsed_ms = (time.perf_counter() - t0) * 1000.0

        self.count_metric.inc()
        self.latency_metric.update(int(elapsed_ms))
        yield result

with beam.Pipeline() as p:
    res = (
        p
        | beam.Create([10, 20, 30, 40, 50])
        | beam.ParDo(InstrumentedDoFn())
    )
    pipeline_result = p.run()
    pipeline_result.wait_until_finish()

metrics = pipeline_result.metrics()
counters = metrics.query(beam.metrics.MetricsFilter().with_name("processed_elements"))["counters"]
for c in counters:
    print(f"Metric Counter [{c.key.metric.name}]: {c.result}")

Metric Counter [processed_elements]: 5


## 5. Concept 2: System Lag vs. Data Freshness

In production Dataflow monitoring, these two signals determine whether to scale compute or investigate upstream sources:

| Scenario | System Lag | Data Freshness | Diagnosis & Action |
| :--- | :--- | :--- | :--- |
| **1. Throughput Bottleneck** | 🔺 Consistently Rising | 🔺 Degraded (Stale) | Processing rate < input rate. Increase `maxNumWorkers` or optimize code. |
| **2. Upstream Idle / Pause** | 🟢 Low / Stable (<5s) | 🔺 Degraded (Stale) | Upstream source has ceased publishing; pipeline is idle and healthy. |
| **3. Traffic Spike** | 🔺 Temporarily Spiking | 🟢 Acceptable | Normal burst; autoscaling will scale out workers to absorb backlog. |
| **4. Healthy Real-Time** | 🟢 Low (<5s) | 🟢 Fresh (<30s) | Pipeline keeping pace with incoming real-time stream. |

## 6. Concept 3: Cloud Monitoring Alert Policies & Dashboards

Let's inspect the production alert policies and dashboard definitions in `monitoring/`:

In [4]:
import json
from pathlib import Path

lag_policy = json.loads(Path("monitoring/alert_policy_system_lag.json").read_text())
print(f"Alert Policy Display Name: {lag_policy['displayName']}")
print(f"Threshold: > {lag_policy['conditions'][0]['conditionThreshold']['thresholdValue']}s")
print(f"Documentation Preview:\n{lag_policy['documentation']['content'][:250]}...")

Alert Policy Display Name: Dataflow SLA: High System Lag (>120s)
Threshold: > 120.0s
Documentation Preview:
### 🚨 Dataflow SLA Alert: Critical System Lag

**Condition**: Pipeline `system_lag` has exceeded 120 seconds for more than 5 minutes.

**Root Cause Diagnostics**:
1. **Throughput Deficit**: The rate of incoming messages from Pub/Sub exceeds worker pr...


## 7. Running the Full Production Benchmark Pipeline Locally

Now let's execute the complete pipeline script [`pipelines/benchmarked_streaming_pipeline.py`](pipelines/benchmarked_streaming_pipeline.py) using `DirectRunner`.

This will process the entire `data/benchmark_telemetry.json` dataset and write performance metrics to `output/`:

In [5]:
import subprocess

cmd = ["bash", "scripts/01_run_local_benchmark.sh"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

 Chapter 05: Running Local Pipeline Benchmark & Step Latency Metrics
Input File:      /Users/zelda.luconi/Medium/google-products/dataflow-workshops/05_streaming_engine_and_monitoring/data/benchmark_telemetry.json
Output Prefix:   /Users/zelda.luconi/Medium/google-products/dataflow-workshops/05_streaming_engine_and_monitoring/output/category_metrics*.json
Simulated Delay: 0.5 ms / record
----------------------------------------------------------------------
----------------------------------------------------------------------
 Execution finished! Generated files in /Users/zelda.luconi/Medium/google-products/dataflow-workshops/05_streaming_engine_and_monitoring/output:
total 8
drwxr-xr-x@ 3 zelda.luconi  staff   96 Oct 27 20:05 .
drwxr-xr-x@ 9 zelda.luconi  staff  288 Oct 27 20:05 ..
-rw-r--r--@ 1 zelda.luconi  staff  924 Oct 27 20:05 category_metrics-00000-of-00001.json

=== Aggregated Category Performance Metrics ===
{"category": "apparel", "transaction_count": 194, "total_revenue": 5

## 8. Inspecting Local Output Files

Let's inspect the aggregated category metrics emitted by the pipeline:

In [6]:
import glob
import json

files = sorted(glob.glob("output/category_metrics*.json"))
print(f"Found {len(files)} output files:")
for f in files:
    with open(f) as fp:
        for line in fp:
            item = json.loads(line.strip())
            print(json.dumps(item, indent=2))

Found 1 output files:
{
  "category": "apparel",
  "transaction_count": 194,
  "total_revenue": 50020.21,
  "avg_revenue": 257.84,
  "avg_step_latency_ms": 0.649,
  "calculated_at": "2026-10-27T19:05:59.035743+00:00"
}
{
  "category": "electronics",
  "transaction_count": 201,
  "total_revenue": 48205.23,
  "avg_revenue": 239.83,
  "avg_step_latency_ms": 0.652,
  "calculated_at": "2026-10-27T19:05:59.037619+00:00"
}
{
  "category": "home_goods",
  "transaction_count": 219,
  "total_revenue": 55559.33,
  "avg_revenue": 253.7,
  "avg_step_latency_ms": 0.661,
  "calculated_at": "2026-10-27T19:05:59.038164+00:00"
}
{
  "category": "software",
  "transaction_count": 183,
  "total_revenue": 43142.23,
  "avg_revenue": 235.75,
  "avg_step_latency_ms": 0.653,
  "calculated_at": "2026-10-27T19:05:59.038591+00:00"
}
{
  "category": "groceries",
  "transaction_count": 203,
  "total_revenue": 49378.39,
  "avg_revenue": 243.24,
  "avg_step_latency_ms": 0.666,
  "calculated_at": "2026-10-27T19:05:59.

## 9. Summary & Next Steps

### What You Learned in Chapter 05:
1. **Dataflow Streaming Engine**: Offloading state and shuffle to dedicated Google backend infrastructure to slash worker VM costs and enable fast autoscaling.
2. **System Lag vs. Data Freshness**: Diagnosing processing deficits versus idle sources.
3. **Step Latency Profiling**: Using `Metrics.distribution` to pinpoint slow transforms.
4. **SLA Alerting**: Automated Cloud Monitoring alert policies on System Lag and Data Freshness.

### Deploying to Google Cloud Dataflow:
To deploy the optimized pipeline and Cloud Monitoring Alert Policies to GCP, run:

```bash
bash scripts/02_run_dataflow_optimized.sh
```

In **Chapter 06**, we will explore **Resilient Operations, Disaster Recovery & Rolling Updates** (In-place `--update`, Pub/Sub seek replay, and zero-downtime maintenance).